# Making Training Fast: compile, AMP & Distributed

P1–P9 got you a correct training loop. This chapter is about the three things that stand
between a correct loop and a *production* one: compilation, mixed precision, and multi-GPU
training. They are also the three things a data scientist moving into MLE has most likely
never touched, because none of them matter until the model stops fitting on one GPU or the
run stops finishing overnight.

**Prerequisite:** P4 (training loop). Conceptual background for the distributed section is in
mlops2.

> ⚠️ Most cells below detect available hardware and degrade gracefully on CPU. The distributed
> section is reference code — it needs `torchrun` and multiple GPUs, so it's written to be
> read and adapted rather than executed inline.

## Why This Matters

- `torch.compile`: what it does, when it pays off, and how it fails
- Mixed precision: fp16 vs bf16, and why bf16 removed the need for loss scaling
- Gradient accumulation to decouple effective batch size from memory
- DDP vs FSDP: the decision rule and what each actually shards
- Reading a profile before optimizing anything

## 1. `torch.compile`

Eager PyTorch dispatches one kernel per operation. For small ops the Python and launch
overhead can exceed the arithmetic — you spend more time *asking* for work than doing it.

`torch.compile` traces your model into a graph, then fuses adjacent operations into single
kernels. A chain like `conv → bias → ReLU` becomes one kernel: one read, one write, instead of
three of each.

```python
model = MyModel().to(device)
model = torch.compile(model)      # that is the entire API
```

### What to expect

| Model shape | Typical benefit |
|---|---|
| Many small ops (pointwise chains, small MLPs) | Large — this is the ideal case |
| Transformer training | Solid, and stacks with other optimizations |
| Big convolutions or matmuls dominating | Modest — already compute-bound |
| Heavy Python control flow in `forward` | Little, and may recompile constantly |

### The costs, which nobody mentions in tutorials

- **Compilation happens on the first batch** and takes seconds to minutes. Never benchmark
  the first iteration.
- **Dynamic shapes trigger recompilation.** Variable sequence lengths or a ragged final batch
  can cause repeated recompiles that cost more than the speedup. Bucket your shapes or pad to
  fixed sizes.
- **Graph breaks** silently reduce the benefit. Data-dependent control flow, `.item()` calls,
  and printing tensors all force the compiler back into eager mode.
- **Debugging gets harder.** Develop in eager, compile once it's correct.

In [ ]:
# Illustrating the mechanism without requiring a GPU: fusion is about memory traffic.
import numpy as np

def unfused(x, w, b):
    """Three passes over memory: one per operation."""
    h = x @ w        # write h
    h = h + b        # read h, write h
    return np.maximum(h, 0)   # read h, write out

def fused(x, w, b):
    """One pass: compute the whole chain per output element while it's in registers."""
    return np.maximum(x @ w + b, 0)

rng = np.random.default_rng(0)
x, w, b = rng.standard_normal((512, 512)), rng.standard_normal((512, 512)), rng.standard_normal(512)
assert np.allclose(unfused(x, w, b), fused(x, w, b))

def traffic(n_elements, n_ops, bytes_per=4):
    """Rough intermediate-tensor traffic: each unfused op reads and writes a full tensor."""
    return n_elements * bytes_per * 2 * n_ops

N = 512 * 512
print(f"{'chain length':>14}{'unfused bytes':>16}{'fused bytes':>14}{'reduction':>12}")
print("-" * 58)
for n_ops in [2, 4, 8, 16]:
    u, f = traffic(N, n_ops), traffic(N, 1)
    print(f"{n_ops:>14}{u/1e6:>14.1f}MB{f/1e6:>12.1f}MB{u/f:>11.0f}x")

print()
print("The longer the pointwise chain, the more there is to fuse. This is why")
print("torch.compile helps most on models with many small operations and least on")
print("models already dominated by one enormous matmul.")

## 2. Mixed Precision

Train with 16-bit tensors for the forward and backward passes while keeping a 32-bit master
copy of the weights for the optimizer update. Roughly half the activation memory and
substantially faster matmuls on tensor cores.

### fp16 vs bf16 — the distinction that matters

| | fp16 | bf16 |
|---|---|---|
| Exponent / mantissa bits | 5 / 10 | 8 / 7 |
| Dynamic range | Narrow — underflows around 6e-8 | Same range as fp32 |
| Precision | Better | Worse |
| **Needs loss scaling?** | **Yes** | **No** |
| Hardware | Broad | Ampere and newer |

Small gradients underflow to zero in fp16 because its exponent range is narrow. Loss scaling
exists to fix this: multiply the loss by a large constant before `backward()` so gradients
land in representable range, then unscale before the optimizer step. `GradScaler` does this
automatically and adapts the factor when it detects overflow.

**bf16 trades mantissa bits for exponent bits and gets fp32's dynamic range.** Gradients don't
underflow, so **loss scaling is unnecessary** — which removes a whole class of "loss became
NaN at step 4000" debugging. On modern hardware, bf16 is the default choice and fp16 is the
compatibility option.

In [ ]:
# Why fp16 needs loss scaling and bf16 does not.
FP16_MIN_SUBNORMAL = 6e-8
FP16_MAX = 65504.0
BF16_MIN_NORMAL = 1.2e-38    # bf16 shares fp32's exponent range

grads = np.array([1e-3, 1e-5, 1e-7, 1e-8, 1e-9, 1e-10])

print(f"{'gradient':>12}{'fp16':>12}{'fp16 x1024':>14}{'bf16':>12}")
print("-" * 52)
for g in grads:
    fp16 = "ok" if g >= FP16_MIN_SUBNORMAL else "UNDERFLOW"
    scaled = g * 1024
    fp16_s = "ok" if FP16_MIN_SUBNORMAL <= scaled <= FP16_MAX else "UNDERFLOW"
    bf16 = "ok" if g >= BF16_MIN_NORMAL else "UNDERFLOW"
    print(f"{g:>12.0e}{fp16:>12}{fp16_s:>14}{bf16:>12}")

print()
print("Columns 2 and 3 are the entire argument for GradScaler: multiplying the loss")
print("shifts small gradients back into fp16's representable window.")
print("Column 4 is the argument for bf16: the problem does not arise.")

In [ ]:
# Reference training loop. NOTE THE IMPORT PATHS — torch.cuda.amp is deprecated;
# the current API is torch.amp with an explicit device argument.
LOOP = r"""
import torch
from torch.amp import autocast, GradScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

# bf16 on Ampere+, else fall back to fp16 (which then needs the scaler).
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

model = MyModel().to(device)
model = torch.compile(model)                       # compile AFTER .to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# GradScaler is a no-op for bf16 - enable it only for fp16.
scaler = GradScaler(device, enabled=(amp_dtype is torch.float16))

ACCUM = 4                                          # effective batch = batch_size * ACCUM

for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad(set_to_none=True)          # set_to_none frees the grad buffers

    for step, (x, y) in enumerate(train_loader):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        with autocast(device_type=device, dtype=amp_dtype):
            loss = criterion(model(x), y)
            loss = loss / ACCUM                    # so the accumulated gradient matches

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM == 0:
            scaler.unscale_(optimizer)             # unscale BEFORE clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
"""
print(LOOP)
print("Three ordering rules that are easy to get wrong:")
print("  1. compile AFTER moving the model to the device.")
print("  2. divide the loss by ACCUM, or your effective learning rate scales with it.")
print("  3. unscale_() BEFORE clip_grad_norm_(), or you clip scaled gradients")
print("     and the clip threshold silently means something different.")

## 3. Gradient Accumulation

Accumulation decouples **effective batch size** from **memory**. Run several forward/backward
passes, letting gradients sum, then step once.

`effective_batch = per_device_batch × accum_steps × world_size`

Two things people get wrong:

1. **Divide the loss by `accum_steps`.** Gradients sum, so without the division your gradient
   magnitude — and therefore your effective learning rate — scales with the accumulation
   count.
2. **It is not free.** You do the same total compute; you just spread it over more steps
   before updating. Throughput drops slightly. It buys memory headroom, not speed.

With BatchNorm there's a subtlety: statistics are computed per micro-batch, so accumulation is
*not* equivalent to a true large batch. LayerNorm and GroupNorm don't have this problem, which
is one more reason transformers are easier to scale.

## 4. DDP vs FSDP

### DDP — Distributed Data Parallel
Every GPU holds a **full replica** of the model. Each processes a different micro-batch. After
backward, an all-reduce averages gradients so all replicas stay identical.

Use when **the model plus optimizer states fit on one GPU.** It's simple, well-tested, and
communication overlaps with backward computation.

### FSDP — Fully Sharded Data Parallel
Shards parameters, gradients, *and* optimizer states across GPUs. Each GPU stores only its
slice, all-gathering the full parameters for a layer just before that layer runs, then
releasing them.

Use when **the model does not fit.** You trade extra communication for a large memory
reduction. FSDP is PyTorch's native implementation of the ZeRO idea (mlops2) — DeepSpeed's
ZeRO stages are the same concept from a different library.

| | DDP | FSDP |
|---|---|---|
| Parameters per GPU | Full copy | 1/N shard |
| Optimizer states | Full copy | 1/N shard |
| Communication | All-reduce gradients | All-gather params + reduce-scatter grads |
| Memory | Highest | Much lower |
| Speed | Faster when it fits | Slower per step, but makes the run possible |
| Use when | Model fits | Model doesn't fit |

**The decision rule:** compute your memory budget first (below). If it fits with room for
activations, use DDP. If not, FSDP.

In [ ]:
def training_memory_gb(params_B, dtype_bytes=2, optimizer="adam",
                       master_weights_fp32=True, world_size=1, sharded=False):
    """
    Per-GPU memory for parameters, gradients, and optimizer states.
    Activations are excluded — they depend on batch size and sequence length.
    """
    p = params_B * 1e9
    weights = p * dtype_bytes
    grads = p * dtype_bytes
    opt = p * 4 * (2 if optimizer == "adam" else 0)      # Adam: fp32 m and v
    master = p * 4 if master_weights_fp32 else 0
    total = weights + grads + opt + master
    if sharded:
        total /= world_size
    return total / 1e9

print("Per-GPU memory for params/grads/optimizer (activations NOT included)\n")
print(f"{'model':>8}{'DDP (1 GPU)':>14}{'FSDP x4':>11}{'FSDP x8':>11}{'FSDP x32':>11}")
print("-" * 56)
for b in [1, 7, 13, 70]:
    ddp = training_memory_gb(b)
    row = "".join(f"{training_memory_gb(b, world_size=n, sharded=True):>11.1f}" for n in [4, 8, 32])
    print(f"{str(b) + 'B':>8}{ddp:>13.1f}G{row}")

print()
print("Read across the 7B row: DDP needs 112 GB per GPU just for state, before a single")
print("activation. It does not fit on an 80 GB card — which is why 'just use DDP'")
print("stops working well below the model size people expect.")
print()
print("Note also what dominates: Adam's two fp32 moments plus the fp32 master copy are")
print("12 bytes/param against 4 for the bf16 weights and gradients. The optimizer is")
print("three quarters of the bill, which is why sharding IT first (ZeRO stage 1) already helps.")

In [ ]:
DDP_REF = r"""
# ---- DDP: launch with `torchrun --nproc_per_node=4 train.py` ----
import os, torch, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def setup():
    dist.init_process_group(backend="nccl")        # nccl for GPU, gloo for CPU
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank

local_rank = setup()
model = MyModel().to(local_rank)
model = DDP(model, device_ids=[local_rank])

sampler = DistributedSampler(dataset)              # each rank sees a disjoint shard
loader = DataLoader(dataset, batch_size=32, sampler=sampler)

for epoch in range(n_epochs):
    sampler.set_epoch(epoch)                       # REQUIRED, or every epoch shuffles identically
    for x, y in loader:
        loss = criterion(model(x.to(local_rank)), y.to(local_rank))
        loss.backward()                            # all-reduce happens here, overlapped
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

if dist.get_rank() == 0:                           # save ONCE, not once per rank
    torch.save(model.module.state_dict(), "ckpt.pt")   # note .module to unwrap DDP
dist.destroy_process_group()
"""

FSDP_REF = r"""
# ---- FSDP: same launcher, different wrapper ----
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from torch.distributed.fsdp import MixedPrecision
import functools

# Wrap per transformer block so all-gather granularity matches compute granularity.
auto_wrap = functools.partial(
    transformer_auto_wrap_policy,
    transformer_layer_cls={MyTransformerBlock},
)

model = FSDP(
    MyModel().to(local_rank),
    auto_wrap_policy=auto_wrap,
    mixed_precision=MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        buffer_dtype=torch.bfloat16,
    ),
    device_id=local_rank,
)
# FSDP handles mixed precision internally - do NOT also wrap the step in autocast/GradScaler.
"""
print(DDP_REF)
print(FSDP_REF)
print("Four failure modes worth memorizing:")
print("  1. Forgetting sampler.set_epoch(epoch)  -> identical shuffling every epoch.")
print("  2. Saving from every rank              -> corrupted or racing checkpoint writes.")
print("  3. Forgetting .module when saving DDP  -> keys prefixed 'module.', load fails.")
print("  4. Uneven batches across ranks         -> a hang, because all-reduce waits forever.")

## 5. Profile Before You Optimize

The most common mistake in this whole chapter is applying it to the wrong bottleneck. If your
GPU is idle 60% of the time waiting on the input pipeline, `torch.compile` buys you nothing.

**Order of investigation:**

1. **Is the GPU actually busy?** `nvidia-smi dmon` or the profiler. Low utilization means data
   loading, not compute.
2. **Fix the data pipeline first.** `num_workers > 0`, `pin_memory=True`,
   `non_blocking=True` transfers, `persistent_workers=True`. This is frequently the entire
   problem.
3. **Then compile and AMP.** Cheap, large, low-risk.
4. **Then batch size and accumulation.** Fill the GPU.
5. **Then distribute.** Only once one GPU is genuinely saturated.

```python
from torch.profiler import profile, ProfilerActivity, schedule

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=schedule(wait=1, warmup=1, active=3),   # skip compile + warmup iterations
    record_shapes=True,
    profile_memory=True,
) as prof:
    for step, (x, y) in enumerate(loader):
        train_step(x, y)
        prof.step()
        if step >= 6:
            break

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
```

> 💡 **Interview Tip:** "I'd profile first" is the answer to almost every performance question,
> and it's only convincing if you can name the tool and what you'd look at. Say: *check GPU
> utilization; if it's low the problem is the input pipeline, not the model.*

## Common Interview Questions

**Q: What does `torch.compile` do, and when does it not help?**
It traces the model into a graph and fuses adjacent operations into single kernels, cutting
kernel-launch overhead and intermediate memory traffic. It helps most on models with many
small operations, least on models already dominated by large matmuls. It hurts when shapes
vary a lot, because each new shape triggers recompilation — bucket or pad your inputs. And
never benchmark the first iteration; that one includes compilation.

**Q: Why does fp16 need loss scaling when bf16 doesn't?**
fp16 has 5 exponent bits, so its dynamic range is narrow and small gradients underflow to
zero. Loss scaling multiplies the loss before backward to shift gradients into representable
range, then unscales before the optimizer step. bf16 has 8 exponent bits — the same range as
fp32 — so underflow doesn't occur and no scaling is needed. bf16 pays for that with fewer
mantissa bits, which turns out to matter far less for training stability.

**Q: DDP or FSDP?**
Compute the memory first: parameters, gradients, and optimizer states. Adam alone is 8
bytes/param for the moments plus 4 for the fp32 master copy, which dominates the bf16 weights.
If that fits on one GPU with room for activations, use DDP — it's simpler and faster because
gradient all-reduce overlaps with backward. If it doesn't fit, FSDP shards all three across
ranks, trading extra communication for the memory you need.

**Q: Is gradient accumulation the same as a larger batch?**
For the gradient, yes — provided you divide the loss by the accumulation count, since
gradients sum. For BatchNorm, no: statistics are still computed per micro-batch, so the
normalization differs from a true large batch. LayerNorm and GroupNorm are unaffected. And
it's not free — same total compute, slightly worse throughput. It buys memory, not speed.

**Q: Training is slow. Walk me through your process.**
Check GPU utilization before anything else. If it's low, the bottleneck is the input pipeline
— fix `num_workers`, `pin_memory`, and transfer overlap first, because no amount of model
optimization helps a starving GPU. If utilization is high, profile to see where time goes,
then apply AMP and `torch.compile`, then raise batch size to fill memory, and only distribute
once a single GPU is genuinely saturated.

**Q: Your DDP job hangs. What are your first guesses?**
Uneven work across ranks — collective operations block until every rank arrives, so if one
rank has fewer batches the others wait forever. Also check that all ranks call the same
collectives in the same order, that nothing is conditionally skipped on rank 0 only, and that
`set_epoch` and sampler configuration are consistent. Debug with the `gloo` backend on CPU
first; the errors are far more legible.

## Key Takeaways
- `torch.compile(model)` fuses kernels; biggest wins on many-small-op models, and dynamic shapes cause costly recompiles
- Use `torch.amp.autocast` / `torch.amp.GradScaler` — `torch.cuda.amp` is the deprecated path
- Prefer bf16: same exponent range as fp32, so no loss scaling and one fewer NaN source
- Enable `GradScaler` only for fp16; it is unnecessary for bf16
- Order matters: compile after `.to(device)`; `unscale_()` before `clip_grad_norm_()`
- Gradient accumulation buys memory, not speed — and requires dividing the loss by the accumulation count
- DDP replicates, FSDP shards; the decision is simply whether params + grads + optimizer states fit
- Adam's optimizer states are ~3× the size of bf16 weights and dominate the training memory budget
- Profile before optimizing: low GPU utilization means the input pipeline, and no kernel fusion fixes that